# WeaveForward — Web Scraper Extraction

**Purpose:** Scrape fiber-composition and clothing-type data directly from the
product pages of **25 sample Philippine clothing brands** and **25 sample multinational
fashion retailers**, parse fiber `%` values, compute EU Ecodesign
biodegradability tiers, and build a `BRAND_FIBER_LOOKUP` table.

**Outputs written to `data/webscraped_data/`:**
- `YYYYMMDD-HHMMSS-webscraped_catalog.csv` — timestamped full product catalog
- `webscraped_catalog.csv` — stable alias (always the latest run)

**Outputs written to `data/processed/`:**
- `YYYYMMDD-HHMMSS-brand_fiber_lookup.json` — timestamped brand fiber profile
- `brand_fiber_lookup.json` — stable alias

In [1]:
import subprocess, sys
from pathlib import Path

PACKAGES = [
    "requests",
    "beautifulsoup4",
    "lxml",
    "pandas",
    "numpy",
    "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])

# ── Path definitions ──────────────────────────────────────────────────────
ROOT          = Path("..").resolve()
DATA_DIR      = ROOT / "data"
PROC_DIR      = DATA_DIR / "processed"
WEB_DIR       = DATA_DIR / "webscraped_data"

for d in [PROC_DIR, WEB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ environment ready")
print(f"  ROOT    : {ROOT}")
print(f"  WEB_DIR : {WEB_DIR}")
print(f"  PROC_DIR: {PROC_DIR}")


✓ environment ready
  ROOT    : D:\School\ISPROJ2\weaveforward-ml
  WEB_DIR : D:\School\ISPROJ2\weaveforward-ml\data\webscraped_data
  PROC_DIR: D:\School\ISPROJ2\weaveforward-ml\data\processed


In [2]:
import json, re, time, random, datetime
from urllib.parse import urlparse, urljoin
import numpy as np
import pandas as pd
import requests
from bs4         import BeautifulSoup
from tqdm        import tqdm
from pathlib     import Path

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}
SCRAPE_TIMEOUT  = 2   # seconds per request
SCRAPE_DELAY = 0  # Slower delay to look more human
MAX_PRODUCTS_PER_COLLECTION = 99999 # <--- CHANGE THIS FOR FULL EXTRACTION

print("✓ imports ready")


✓ imports ready


---
## 1-A · Brand Catalogue — Philippine & Multinational Retailers

Each brand entry holds only the **root / homepage URL**.  
`discover_collection_urls()` (1-B-3) crawls that root to find all clothing-category
and collection sub-pages automatically, so no paths are hardcoded.

In [3]:
# ── 1-A  Brand catalogue ──────────────────────────────────────────────────
# Each entry: (brand_name, root_url, source_region, country)
# root_url is the brand homepage or top-level shop root.
# Collection sub-pages are discovered automatically by discover_collection_urls().

PH_BRANDS = [
    ("OXGN", "https://www.oxgnfashion.com/", "philippine", "PH"),
    ("Memo", "https://www.memofashion.com", "philippine", "PH"),
    ("Pull&Bear", "https://www.pullandbear.com/ph/", "philippine", "PH"),
    ("The Big Silence", "https://thebigsilence.com/", "philippine", "PH"),
    ("Tropa Store", "https://tropastore.com/", "philippine", "PH"),
    ("Patton Studio", "https://pattonstudio.com/", "philippine", "PH"),
    ("Levi's", "https://levi.com.ph/", "philippine", "PH"),
    ("Urban Outfitters", "https://www.urbanoutfitters.com/", "philippine", "PH"),
    ("Undershirtguy",     "https://undershirtguy.com/",          "international", "GLOBAL"),
    ("Rokman",     "https://rokmanwaterproof.com/",             "international", "GLOBAL"),
    ("Noti But We",     "https://notibutwe.com/",             "philippine", "PH"),
    ("Apara Studio",     "https://www.apara-studio.com/",             "international", "GLOBAL"),
    ("Guess",     "https://guess.com.ph/",             "philippine", "PH"),
    ("Phoebe Philo",     "https://www.phoebephilo.com/",             "international", "GLOBAL"),  
    ("The Editor's Market", "http://ph.theeditorsmarket.com/", "philippine", "PH"),
   
    ("Regatta", "https://www.regattalifestyle.com", "philippine", "PH"),
    ("ForMe", "https://www.formeclothing.com", "philippine", "PH"),
    ("Bench", "https://shop.bench.com.ph/", "philippine", "PH"),
    ("World Balance", "https://worldbalance.com.ph/", "philippine", "PH"),
    ("Old Navy", "https://oldnavy.com.ph/", "philippine", "PH"),
    ("GQ", "https://www.gq-magazine.co.uk/gallery/white-t-shirt-every-man-needs-one", "philippine", "PH"),
    ("No Emotions", "https://www.noemotions.co.uk/", "philippine", "PH"),
    ("Frank and Eileen", "https://www.frankandeileen.com/", "philippine", "PH"),
    ("Made by Fade", "https://madebyfade.com/", "philippine", "PH"),
    ("Gaui", "https://shopgaui.com/", "philippine", "PH"),
    ("Just G", "https://justg.com.ph/", "philippine", "PH"),
    ("Sola", "https://sola.com.ph/", "philippine", "PH"),
    ("Gymshark",          "https://www.gymshark.com/",              "international", "GLOBAL"),
    ("Adored Vintage",    "https://adoredvintage.com/",             "international", "GLOBAL"),
    ("Tayo Studio",     "https://shoptayostudio.com/",             "philippine", "PH"),
    ("Linya Linya",     "https://linyalinya.ph/",             "philippine", "PH"),
    ("Princess Polly",     "https://us.princesspolly.com/",             "international", "GLOBAL"),
    ("Coastal Bloom",     "https://coastalbloom.com/",             "international", "GLOBAL"),
]

MULTINATIONAL_BRANDS = [
]

ALL_BRANDS = PH_BRANDS + MULTINATIONAL_BRANDS
print(f"✓ {len(PH_BRANDS)} Philippine brands + {len(MULTINATIONAL_BRANDS)} multinational = {len(ALL_BRANDS)} total")

✓ 33 Philippine brands + 0 multinational = 33 total


---
## 1-B · Live Scraper — requests + BeautifulSoup

### 1-B-1 · Fiber Pattern Parser & Clothing-Type Detector


In [4]:
# ── Regex & mapping constants ─────────────────────────────────────────────
# Fiber composition regex — matches patterns like "95% Cotton" or "100% Polyester"
FIBER_RE = re.compile(
    r'(\d{1,3})\s*%\s*(cotton|polyester|nylon|wool|linen|silk|rayon|viscose|'
    r'acrylic|elastane|spandex|lycra|modal|bamboo|hemp|denim|cashmere|tencel|'
    r'lyocell|'
    r'alpaca)',
    re.IGNORECASE,
)

# Keyword → canonical clothing category mapping
CLOTHING_TYPE_MAP = {
    "dress": "dress",       "dresses": "dress",
    "tee": "t-shirt",       "t-shirt": "t-shirt",   "tshirt": "t-shirt",
    "polo": "polo",
    "jacket": "jacket",     "coat": "jacket",
    "jeans": "jeans",
    "suit": "suit",         "blazer": "blazer",
    "swimwear": "swimwear", "swim": "swimwear",
    "hoodie": "hoodie",     "sweatshirt": "hoodie",
    "knit": "knitwear",     "sweater": "knitwear",  "pullover": "knitwear",
    

    "top": "top",           "tops": "top",
    "shirt": "shirt",       "shirts": "shirt",      "blouse": "shirt",
    
    "pants": "pants",       "trousers": "pants",    "slacks": "pants",
    "skirt": "skirt",       "skirts": "skirt",
    "shorts": "shorts",
    "underwear": "underwear", "bra": "underwear",
    "activewear": "activewear", "leggings": "activewear",
    "denim": "jeans",
}

print("✓ FIBER_RE and CLOTHING_TYPE_MAP ready")


✓ FIBER_RE and CLOTHING_TYPE_MAP ready


In [5]:
def detect_clothing_type(title: str, url: str = "", clean_body: str = "") -> str:
    """
    ZERO-TOLERANCE DETECTION: 
    Prioritizes Title > URL.
    Completely avoids Full Body Text for keywords to prevent "Dress/Polo" noise.
    """
    # 1. Check Title (Most Reliable)
    t_clean = title.lower()
    for kw, cat in CLOTHING_TYPE_MAP.items():
        if re.search(fr"\b{re.escape(kw)}\b", t_clean):
            return cat
            
    # 2. Check URL (Secondary Reliability)
    u_clean = url.lower()
    for kw, cat in CLOTHING_TYPE_MAP.items():
        if re.search(fr"\b{re.escape(kw)}\b", u_clean):
            return cat
            
    return "unspecified"
print("✓ detect_clothing_type() ready")


✓ detect_clothing_type() ready


In [6]:
def parse_fiber_composition(text: str) -> dict:
    """
    The "Smart-Break" Parser: 
    Supports complex blends but stops the second it hits 100%
    to prevent double-counting repeated text.
    """
    fibers: dict = {}
    if not text: return fibers
    
    for m in FIBER_RE.finditer(text):
        pct = float(m.group(1))
        fib = (m.group(2).lower()
               .replace("spandex",   "elastane")
               .replace("lycra",     "elastane")
               .replace("polyamide", "nylon")
               .replace("microfiber","microfibre"))
        
        # ─── THE SMART BREAK ─────────────────────────────────────────
        # If we already have 90%+ and we see a duplicate material,
        # then it's a repeated description section. STOP looking.
        if sum(fibers.values()) >= 90 and fib in fibers:
            break
        # ─────────────────────────────────────────────────────────────
            
        fibers[fib] = fibers.get(fib, 0.0) + pct
        
    total = sum(fibers.values())
    
    # ─── THE STRICT TRUTH ──────────────────────────────────────────
    # No scaling, no rounding. If it's 101 or 99, we drop it.
    if total == 100:
        return fibers
    # ───────────────────────────────────────────────────────────────
    
    return {} # Drops the product


### 1-B-2 · Page Scraper Function


In [7]:
def scrape_page(url, brand, source, country):
    records = []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, "lxml")
        product_links = set()
        for a in soup.find_all("a", href=True):
            if any(kw in a["href"].lower() for kw in ["/products/", "/p/", "/item/"]):
                product_links.add(urljoin(url, a["href"]))
        
        pages_to_parse = list(product_links)[:MAX_PRODUCTS_PER_COLLECTION] if product_links else [url]
        for purl in pages_to_parse: 
            try:
                time.sleep(SCRAPE_DELAY)
                p_resp = requests.get(purl, headers=HEADERS, timeout=10)
                p_soup = BeautifulSoup(p_resp.text, "lxml")
                
                title      = p_soup.find("h1").get_text(strip=True) if p_soup.find("h1") else brand
                page_text  = p_soup.get_text(" ", strip=True) 
                fibers     = parse_fiber_composition(page_text)

                # Capture a short snippet of raw fabric text for auditability
                fab_text = ""
                for m in FIBER_RE.finditer(page_text):
                    start    = max(0, m.start() - 60)
                    end      = min(len(page_text), m.end() + 60)
                    fab_text = page_text[start:end].strip()[:200]
                    break
                
                if not fibers: continue
                
                record = {
                    "brand": brand,
                    "product_name": title,
                    "clothing_type": detect_clothing_type(title, purl, page_text),
                    "fabric_composition":  fab_text,
                    "fiber_json": json.dumps(fibers),
                    "most_dominant_fiber": max(fibers, key=fibers.get),
                    "source": source,
                    "country_of_brand": country,
                    "scraped_url": purl,
                    "scraped_at": datetime.datetime.now().isoformat(),
                    "origin": "live",
                }
                if record["clothing_type"] == "unspecified":
                    print("Skipping unspecified clothing type.")
                    continue
                # --- LIVE MONITORING ---
                print(f"    [OK] {record['brand']} | {record['clothing_type'].upper()} | {record['product_name']}")
                print(f"         Fibers: {record['fiber_json']}")
                
                records.append(record)
            except: continue
    except: pass
    return records

### 1-B-3 · Collection URL Discovery

`discover_collection_urls(root_url)` crawls the brand's homepage and returns
all clothing-category / collection listing pages it finds in `<a href>` links —
no hardcoded paths required. The run-scraper cell feeds these discovered URLs
into `scrape_page()` instead of a single fixed URL.

In [8]:
# ── Private regex constants for discover_collection_urls() ────────────────
# Path segments that indicate a clothing collection / category page
_COLLECTION_RE = re.compile(
    r'/('
    r'collections?|categories?|category|'
    r'clothing|clothes|apparel|'
    r'c/|browse/|shop/|products?/|'
    r'women|mens?|ladies|girls?|boys?|kids?|'
    r'tops?|shirts?|blouses?|dresses?|skirts?|'
    r'pants?|trousers?|jeans?|bottoms?|'
    r'jackets?|coats?|outerwear|knitwear|sweaters?|hoodies?|'
    r'activewear|swimwear|underwear|lingerie'
    r')',
    re.IGNORECASE,
)

# Path segments that are definitely NOT product listings
_SKIP_RE = re.compile(
    r'/(cart|checkout|account|login|register|search|wishlist|'
    r'faq|about|contact|blog|press|careers?|stores?|'
    r'sustainability|privacy|returns?|size.?guide|gift)',
    re.IGNORECASE,
)

print("✓ _COLLECTION_RE and _SKIP_RE ready")


✓ _COLLECTION_RE and _SKIP_RE ready


In [9]:
def discover_collection_urls(root_url: str, max_collections: int = 8) -> list[str]:
    """
    Crawl root_url (brand homepage) and return up to max_collections
    clothing-category / collection listing URLs found in <a href> links.

    Strategy:
      1. GET root_url.
      2. Walk every <a href> — resolve relative URLs, keep same-domain only.
      3. Filter by _COLLECTION_RE and exclude _SKIP_RE noise links.
      4. Sort by number of clothing keywords in the path (most specific first).
      5. Return up to max_collections unique URLs.
      6. Fallback: if nothing is found, return [root_url] so scrape_page()
         still has something to work with.
    """
    parsed_root = urlparse(root_url)
    base        = f"{parsed_root.scheme}://{parsed_root.netloc}"

    try:
        resp = requests.get(root_url, headers=HEADERS, timeout=SCRAPE_TIMEOUT)
        if resp.status_code != 200:
            return [root_url]
        soup = BeautifulSoup(resp.text, "lxml")
    except Exception:
        return [root_url]

    seen:       set  = set()
    candidates: list = []

    for a in soup.find_all("a", href=True):
        raw_href = a["href"].strip()
        if not raw_href or raw_href.startswith(("#", "javascript", "mailto")):
            continue

        # Resolve to absolute URL; strip query-string and trailing slash
        full = (raw_href if raw_href.startswith("http") else urljoin(base, raw_href))
        full = full.split("?")[0].split("#")[0].rstrip("/")

        # Keep same domain only
        if urlparse(full).netloc != parsed_root.netloc:
            continue

        path = urlparse(full).path.lower()

        if _SKIP_RE.search(path):
            continue
        if not _COLLECTION_RE.search(path):
            continue
        if full in seen:
            continue

        seen.add(full)
        candidates.append(full)

    # Rank: paths that contain more clothing-type keywords score higher
    _RANK_WORDS = {
        "tops", "shirts", "dresses", "clothing", "collections",
        "women", "men", "ladies", "apparel", "clothes",
    }
    candidates.sort(
        key=lambda u: sum(w in u.lower() for w in _RANK_WORDS),
        reverse=True,
    )

    result = candidates[:max_collections]
    if not result:
        result = [root_url]

    return result


print("✓ discover_collection_urls() ready")


✓ discover_collection_urls() ready


### 1-B-4 · Run Live Scraper

For each brand, `discover_collection_urls()` crawls the root homepage and
returns up to 8 clothing-category pages. `scrape_page()` is then called on
each discovered page to harvest product detail links and extract fiber data.  
Brands whose sites require JavaScript or block requests will yield 0 rows.

> **Testing mode:** Currently running with **2 Philippine + 2 multinational brands** (`TEST_BRANDS`).  
> To run the full 50-brand scrape, replace `TEST_BRANDS` with `ALL_BRANDS` in the scraper cell below.


In [10]:
# ── 1-B-4  Run live scraper across brand root URLs ────────────────────────
#
# TEST_MODE   = True  → scrape 2 PH + 2 multinational brands only
# TEST_MODE   = False → scrape all 50 brands (full production run)
#
TEST_MODE   = False
TEST_BRANDS = PH_BRANDS[:2] + MULTINATIONAL_BRANDS[:2]
RUN_BRANDS  = TEST_BRANDS if TEST_MODE else ALL_BRANDS

if TEST_MODE:
    print(f"⚠ TEST MODE — scraping {len(RUN_BRANDS)} brands "
          f"({len(PH_BRANDS[:2])} PH + {len(MULTINATIONAL_BRANDS[:2])} multinational):")
    for b in RUN_BRANDS:
        print(f"  • {b[0]}  ({b[3]})")
    print()
else:
    print(f"PRODUCTION MODE — scraping all {len(RUN_BRANDS)} brands")
    print()

_SCHEMA = [
    "brand", "product_name", "clothing_type", "fabric_composition",
    "fiber_json", "most_dominant_fiber", "source", "country_of_brand",
    "scraped_url", "scraped_at", "origin",
]

live_rows = []

try:
    for brand_name, root_url, source, country in tqdm(RUN_BRANDS, desc="Scraping brands"):
        collection_urls = discover_collection_urls(root_url)
        brand_rows = []
        for col_url in collection_urls:
            rows = scrape_page(col_url, brand_name, source, country)
            brand_rows.extend(rows)
        live_rows.extend(brand_rows)
        if brand_rows:
            print(f"  ✓ {brand_name:30s} {len(brand_rows):3d} products  "
                  f"({len(collection_urls)} collection pages crawled)")
except KeyboardInterrupt:
    print(f"\n⚠ Scrape interrupted — {len(live_rows)} rows collected so far will be saved.")

# ── Full 50-brand loop (kept for production use — set TEST_MODE = False above) ──
# try:
#     for brand_name, root_url, source, country in tqdm(ALL_BRANDS, desc="Scraping brands"):
#         collection_urls = discover_collection_urls(root_url)
#         brand_rows = []
#         for col_url in collection_urls:
#             rows = scrape_page(col_url, brand_name, source, country)
#             brand_rows.extend(rows)
#         live_rows.extend(brand_rows)
#         if brand_rows:
#             print(f"  ✓ {brand_name:30s} {len(brand_rows):3d} products  "
#                   f"({len(collection_urls)} collection pages crawled)")
# except KeyboardInterrupt:
#     print(f"\n⚠ Scrape interrupted — {len(live_rows)} rows collected so far will be saved.")

print(f"\n  Live scrape total: {len(live_rows)} rows from "
      f"{len({r['brand'] for r in live_rows})} brands")

df_catalog = (
    pd.DataFrame(live_rows, columns=_SCHEMA)
    if live_rows
    else pd.DataFrame(columns=_SCHEMA)
)
df_catalog = df_catalog.drop_duplicates(subset=["brand", "product_name"])

print(f"\n✓ df_catalog: {len(df_catalog):,} rows | {df_catalog['brand'].nunique()} brands")
print(f"\n  Clothing-type distribution:")
for ct, n in df_catalog["clothing_type"].value_counts().head(10).items():
    print(f"    {ct:<16} {n:>4}")

PRODUCTION MODE — scraping all 33 brands



Scraping brands:   0%|          | 0/33 [00:00<?, ?it/s]

    [OK] OXGN | T-SHIRT | Ringer Raglan Graphic Slim Cropped T-Shirt
         Fibers: {"cotton": 90.0, "elastane": 10.0}
    [OK] OXGN | T-SHIRT | Graphic T-Shirt
         Fibers: {"polyester": 65.0, "cotton": 35.0}
    [OK] OXGN | T-SHIRT | Graphic T-Shirt
         Fibers: {"polyester": 65.0, "cotton": 35.0}
    [OK] OXGN | DRESS | Square Neck Sleeveless Dress
         Fibers: {"cotton": 55.0, "polyester": 38.0, "elastane": 7.0}
    [OK] OXGN | T-SHIRT | Ringer Striped T-Shirt
         Fibers: {"polyester": 65.0, "cotton": 35.0}
    [OK] OXGN | DRESS | Raglan Polo Dress
         Fibers: {"polyester": 65.0, "cotton": 35.0}
Skipping unspecified clothing type.
    [OK] OXGN | TOP | Jersey Boxy Lace Top
         Fibers: {"polyester": 100.0}
    [OK] OXGN | PANTS | Printed Barrel Trousers
         Fibers: {"cotton": 100.0}
    [OK] OXGN | T-SHIRT | Jersey Graphic Slim T-Shirt
         Fibers: {"polyester": 100.0}
    [OK] OXGN | TOP | Jersey Graphic Boxy Top
         Fibers: {"polyester": 

Scraping brands:   3%|▎         | 1/33 [03:15<1:44:01, 195.05s/it]

    [OK] OXGN | SHORTS | Baggy Denim Shorts
         Fibers: {"cotton": 100.0}
  ✓ OXGN                           273 products  (8 collection pages crawled)
    [OK] Memo | POLO | Textured Knit Polo
         Fibers: {"viscose": 100.0}
    [OK] Memo | T-SHIRT | 3/4 Flat Knit T-Shirt
         Fibers: {"viscose": 100.0}
    [OK] Memo | T-SHIRT | Textured T-Shirt with Openwork Detail
         Fibers: {"viscose": 100.0}
    [OK] Memo | POLO | V-Placket Polo with Openwork Detail
         Fibers: {"viscose": 100.0}
    [OK] Memo | POLO | V-Placket Polo with Openwork Detail
         Fibers: {"viscose": 100.0}
    [OK] Memo | SHIRT | Puff Sleeve Shirt
         Fibers: {"cotton": 60.0, "polyester": 40.0}
    [OK] Memo | JACKET | Flat Knit Short Sleeve Jacket
         Fibers: {"viscose": 100.0}
    [OK] Memo | TOP | Textured Cardigan
         Fibers: {"viscose": 100.0}
    [OK] Memo | SHIRT | Y-Placket Linen Shirt
         Fibers: {"cotton": 97.0, "elastane": 3.0}
    [OK] Memo | SHIRT | Eyelet S

...

Skipping unspecified clothing type.
  ✓ Linya Linya                     16 products  (8 collection pages crawled)
    [OK] Princess Polly | DRESS | Loulah
Sheer
Chiffon
Maxi
Dress
Cream
Polka
Dot
         Fibers: {"polyester": 100.0}
Skipping unspecified clothing type.
    [OK] Princess Polly | TOP | Lailana
Tie
Back
Strapless
Scarf
Top
Burgundy
/
Blue
Stripe
         Fibers: {"polyester": 100.0}
Skipping unspecified clothing type.
    [OK] Princess Polly | POLO | Highlights
Jersey
Polo
Top
Multi
Stripe
         Fibers: {"polyester": 95.0, "elastane": 5.0}
    [OK] Princess Polly | JEANS | Tag
Longline
Denim
Jorts
Washed
Camo
         Fibers: {"cotton": 90.0, "viscose": 10.0}
Skipping unspecified clothing type.
    [OK] Princess Polly | SKIRT | Natana
Asymmetrical
Midi
Skirt
Multi
Check
         Fibers: {"polyester": 100.0}
Skipping unspecified clothing type.
    [OK] Princess Polly | DRESS | Mi
Cherie
Plunge
Ruched
Mini
Dress
Apricot
         Fibers: {"polyester": 100.0}
    [OK] Prin

Scraping brands:  97%|█████████▋| 32/33 [1:55:51<03:42, 222.30s/it]

  ✓ Princess Polly                  72 products  (8 collection pages crawled)
    [OK] Coastal Bloom | TOP | Copacabana Coast Graphic Italian Top
         Fibers: {"cotton": 100.0}
    [OK] Coastal Bloom | TOP | Whispering Ruffle Italian Sleeveless Top- Beige
         Fibers: {"viscose": 50.0, "linen": 50.0}
    [OK] Coastal Bloom | PANTS | Relaxed Draped Front Denim Pants
         Fibers: {"lyocell": 100.0}
    [OK] Coastal Bloom | TOP | Strawberry Crush Graphic Italian Top
         Fibers: {"cotton": 100.0}
    [OK] Coastal Bloom | DRESS | Effortless Impact Italian Dress- Pink
         Fibers: {"linen": 100.0}
    [OK] Coastal Bloom | PANTS | Bohemian Lace Denim Pants
         Fibers: {"cotton": 64.0, "polyester": 32.0, "viscose": 3.0, "elastane": 1.0}
    [OK] Coastal Bloom | KNITWEAR | Olive & Lemon St Tropez Short Sleeve Sweater
         Fibers: {"polyester": 65.0, "viscose": 35.0}
    [OK] Coastal Bloom | DRESS | Blush Garden Italian Button Dress- Light Blue
         Fibers: {"li

Scraping brands: 100%|██████████| 33/33 [2:02:09<00:00, 222.10s/it]

    [OK] Coastal Bloom | DRESS | Court Contrast Trim Shift Dress - Magenta
         Fibers: {"polyester": 80.0, "cotton": 20.0}
  ✓ Coastal Bloom                  367 products  (8 collection pages crawled)

  Live scrape total: 3439 rows from 31 brands

✓ df_catalog: 1,381 rows | 31 brands

  Clothing-type distribution:
    top               300
    t-shirt           269
    dress             168
    shirt             163
    pants              85
    knitwear           63
    hoodie             60
    polo               55
    jeans              46
    shorts             44


---
## 1-C · Biodegradability Tier Classification — BRAND_FIBER_LOOKUP

### Regulatory References

The biodegradability tiers applied to each product's fiber composition are
derived from two authoritative sources:

**1. EU Regulation 2024/1781 — Ecodesign for Sustainable Products (ESPR)**
> *Regulation (EU) 2024/1781 of the European Parliament and of the Council,*
> Official Journal of the European Union, 2024.
> Annex I (textile product groups) and the accompanying Commission staff
> working document on textile sustainability scoring methodology.
> The regulation establishes minimum recycled-content and natural-fiber
> thresholds for product sustainability labelling across EU member states.
> Delegated acts specifying exact numeric thresholds for textile
> biodegradability scoring are ongoing as of 2024–2026.

**2. GOTS v6.0 — Global Organic Textile Standard**
> *Global Organic Textile Standard, Version 6.0*, GOTS, 2020.
> Establishes **≥ 85 % certified organic natural fibers** as the minimum
> threshold for main-label GOTS certification — the origin of the 85 %
> boundary used in this notebook.

### Tier Mapping Applied in This Notebook

| Tier | Bio-fiber share | Interpretation |
|------|----------------|----------------|
| **high** | ≥ 85 % | Predominantly natural / biodegradable — GOTS-aligned |
| **medium** | 50 – 84 % | Mixed composition |
| **low** | < 50 % | Synthetic-dominant |

> ⚠ **Note:** The exact numeric thresholds in the ESPR delegated textile act
> are still being finalised. The 85 / 50 split is an evidence-based
> approximation aligned with GOTS v6.0 and the draft ESPR textile methodology.

---

### 1-C-1 · Bio-Share & Tier Helper Functions

In [11]:
# ── 1-C-1  Bio-fiber vocabulary ─────────────────────────────────────────

BIO_FIBERS = frozenset([
    "cotton", "linen", "hemp", "wool", "silk",
    "bamboo", "tencel", "lyocell", "modal",
    "cashmere", "viscose", "rayon", "acetate", "denim",
])


def bio_share(fibers: dict) -> float:
    """Return the percentage of bio/natural fibers in a fiber-composition dict."""
    total = sum(fibers.values())
    if total == 0:
        return 0.0
    return round(sum(v for k, v in fibers.items() if k in BIO_FIBERS) / total * 100, 2)


def biodeg_tier(bio_pct: float) -> str:
    """Map a bio-fiber percentage to an EU-Ecodesign-aligned biodegradability tier.

    Thresholds:
        ≥ 85 % → 'high'   (GOTS v6.0 main-label threshold)
        ≥ 50 % → 'medium'
        < 50 % → 'low'
    """
    if bio_pct >= 85:
        return "high"
    if bio_pct >= 50:
        return "medium"
    return "low"


print("✓ BIO_FIBERS vocabulary loaded:", len(BIO_FIBERS), "fiber types")
print("✓ bio_share() and biodeg_tier() ready")

✓ BIO_FIBERS vocabulary loaded: 14 fiber types
✓ bio_share() and biodeg_tier() ready


### 1-C-2 · Annotate Product Catalog

Parse the `fiber_json` column of `df_catalog` into Python dicts, then compute
`fs_bio_share` (% bio-fiber) and `fs_biodeg_tier` for every row.

In [12]:
# ── §1-C-2  Parse fiber_json → annotate df_catalog ───────────────────────

df_catalog["fiber_dict"] = df_catalog["fiber_json"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else {}
)
df_catalog["fs_bio_share"]   = df_catalog["fiber_dict"].apply(bio_share)
df_catalog["fs_biodeg_tier"] = df_catalog["fs_bio_share"].apply(biodeg_tier)

print(f"✓ Annotated {len(df_catalog):,} rows")
print("\n  Biodegradability tier distribution (EU Ecodesign 2024/1781 / GOTS v6.0):")
for tier, cnt in df_catalog["fs_biodeg_tier"].value_counts().items():
    print(f"    {tier:<8} {cnt:>5}  ({cnt / len(df_catalog) * 100:.1f} %)")

print("\n  Sample rows:")
print(df_catalog[["brand", "clothing_type", "most_dominant_fiber",
                   "fs_bio_share", "fs_biodeg_tier", "source"]].head(8).to_string(index=False))

✓ Annotated 1,381 rows

  Biodegradability tier distribution (EU Ecodesign 2024/1781 / GOTS v6.0):
    high       778  (56.3 %)
    low        401  (29.0 %)
    medium     202  (14.6 %)

  Sample rows:
brand clothing_type most_dominant_fiber  fs_bio_share fs_biodeg_tier     source
 OXGN       t-shirt              cotton          90.0           high philippine
 OXGN       t-shirt           polyester          35.0            low philippine
 OXGN         dress              cotton          55.0         medium philippine
 OXGN       t-shirt           polyester          35.0            low philippine
 OXGN         dress           polyester          35.0            low philippine
 OXGN           top           polyester           0.0            low philippine
 OXGN         pants              cotton         100.0           high philippine
 OXGN       t-shirt           polyester           0.0            low philippine


### 1-C-3 · Aggregate Per-Brand Median Profile → BRAND_FIBER_LOOKUP

For each brand, compute the **median fiber share** across all scraped products.
Fibers contributing < 2 % at the median are dropped to keep the profile clean.
The resulting dict `BRAND_FIBER_LOOKUP[brand]` holds:
`fibers`, `bio_share`, `biodeg_tier`, `item_count`, `source`.

In [13]:
# ── 1-C-3  Per-brand median fiber profile ───────────────────────────────

_brand_records = []
for brand, grp in df_catalog[df_catalog["fiber_dict"].apply(bool)].groupby("brand"):
    agg: dict = {}
    for fdict in grp["fiber_dict"]:
        for k, v in fdict.items():
            agg.setdefault(k, []).append(v)

    # median share per fiber; drop fibers with median ≤ 2 %
    brand_fiber = {
        k: round(float(np.median(v)), 1)
        for k, v in agg.items()
        if np.median(v) > 2
    }

    # re-normalise to 100 %
    total = sum(brand_fiber.values())
    if total > 0:
        brand_fiber = {k: round(v / total * 100, 1) for k, v in brand_fiber.items()}

    _brand_records.append({
        "brand":       brand.lower().strip(),
        "fibers":      brand_fiber,
        "bio_share":   bio_share(brand_fiber),
        "biodeg_tier": biodeg_tier(bio_share(brand_fiber)),
        "item_count":  len(grp),
        "source":      grp["source"].iloc[0],
    })

BRAND_FIBER_LOOKUP: dict = {r["brand"]: r for r in _brand_records}

print(f"✓ BRAND_FIBER_LOOKUP built: {len(BRAND_FIBER_LOOKUP)} brands")
print("\n  Sample entries:")
for brand, rec in list(BRAND_FIBER_LOOKUP.items())[:4]:
    print(f"    {brand:<22} tier={rec['biodeg_tier']:<8} bio={rec['bio_share']:5.1f}%  "
          f"items={rec['item_count']:>3}  fibers={rec['fibers']}")

✓ BRAND_FIBER_LOOKUP built: 31 brands

  Sample entries:
    adored vintage         tier=medium   bio= 77.8%  items= 39  fibers={'linen': 8.9, 'cotton': 18.6, 'nylon': 5.8, 'elastane': 0.9, 'rayon': 18.6, 'polyester': 15.5, 'viscose': 13.0, 'wool': 18.6}
    apara studio           tier=high     bio=100.0%  items= 15  fibers={'cotton': 100.0}
    bench                  tier=low      bio= 28.7%  items= 22  fibers={'nylon': 22.2, 'elastane': 3.5, 'cotton': 26.1, 'polyester': 19.6, 'linen': 2.6, 'acrylic': 26.1}
    coastal bloom          tier=high     bio= 86.9%  items=123  fibers={'cotton': 14.8, 'viscose': 10.9, 'linen': 15.6, 'lyocell': 11.7, 'polyester': 7.8, 'elastane': 0.6, 'modal': 7.8, 'tencel': 13.2, 'rayon': 12.9, 'nylon': 4.7}


---
## 1-D · Historical Archive — Discontinued Item Tracking

Every time the scraper runs it produces a fresh snapshot, but items that were
once present and have since been removed from brand catalogues disappear silently.
This section maintains a **cumulative archive** (`webscraped_catalog_archive.csv`)
that persists across runs and tracks each product's full lifecycle via three
provenance columns:

| Column | Meaning |
|---|---|
| `first_scraped_at` | ISO timestamp of the first scrape run that found this product |
| `last_seen_at` | ISO timestamp of the most recent run that found this product |
| `is_active` | `True` if found in the current run; `False` = not seen (possibly discontinued) |

**Match key:** `(brand, product_name)` — a returning product updates `last_seen_at`
and reactivates `is_active`; a product absent from the current run is automatically
flipped to `is_active = False`.


In [14]:
# ── 1-D  Historical product archive — merge new scrape into running catalog ──
#
# ARCHIVE: data/webscraped_data/webscraped_catalog_archive.csv
#   • Persists every product ever seen across all scrape runs.
#   • Adds three provenance columns: first_scraped_at, last_seen_at, is_active.
#   • is_active = True  → item appeared in the most recent scrape run.
#   • is_active = False → item was NOT seen in the most recent run (possibly discontinued).
#
# Match key: (brand, product_name) — same product has a stable scraper-derived name.
# Deduplication: the same product can surface on multiple collection pages in one
# scrape run (e.g., a shirt found under both /collections/women and /collections/tops).
# df_current is deduplicated on (brand, product_name) before any merge logic,
# keeping the last occurrence (most recently parsed page).  A final guard dedup is
# applied to df_archive_updated so that malformed inputs from past runs cannot
# accumulate duplicate keys across multiple scrape cycles.

ARCHIVE_PATH = WEB_DIR / "webscraped_catalog_archive.csv"

_archive_cols = [
    "brand", "product_name", "clothing_type", "fabric_composition",
    "fiber_json", "most_dominant_fiber", "fs_bio_share", "fs_biodeg_tier",
    "source", "country_of_brand", "scraped_url", "first_scraped_at",
    "last_seen_at", "origin", "is_active",
]

_MATCH_KEY = ["brand", "product_name"]

# ── Load existing archive (or initialise empty on first run) ─────────────
if ARCHIVE_PATH.exists():
    df_archive = pd.read_csv(ARCHIVE_PATH, dtype=str)
    # Guard: deduplicate any pre-existing duplicate keys in the loaded archive
    _before = len(df_archive)
    df_archive = df_archive.drop_duplicates(subset=_MATCH_KEY, keep="last").reset_index(drop=True)
    _dupes_in_archive = _before - len(df_archive)
    print(f"✓ Loaded existing archive: {len(df_archive):,} rows | "
          f"{df_archive['brand'].nunique()} brands"
          + (f"  (removed {_dupes_in_archive} pre-existing duplicates)" if _dupes_in_archive else ""))
else:
    df_archive = pd.DataFrame(columns=_archive_cols)
    print("✓ No existing archive — initialising fresh archive.")

# ── Prepare current-run rows for merge ───────────────────────────────────
_current_save_cols = [
    "brand", "product_name", "clothing_type", "fabric_composition",
    "fiber_json", "most_dominant_fiber", "fs_bio_share", "fs_biodeg_tier",
    "source", "country_of_brand", "scraped_url", "scraped_at", "origin",
]
df_current = df_catalog[_current_save_cols].copy()
df_current["first_scraped_at"] = df_current["scraped_at"]
df_current["last_seen_at"]     = df_current["scraped_at"]
df_current["is_active"]        = True
df_current = df_current.drop(columns=["scraped_at"], errors="ignore")
df_current = df_current.reindex(columns=_archive_cols)

# ── Intra-run deduplication ───────────────────────────────────────────────
# The scraper visits multiple collection pages per brand; the same product
# (same brand + product_name) can appear more than once in a single run.
# Keep the last occurrence so the most recently parsed fiber data wins.
_before_dedup = len(df_current)
df_current = df_current.drop_duplicates(subset=_MATCH_KEY, keep="last").reset_index(drop=True)
_intra_dupes = _before_dedup - len(df_current)
if _intra_dupes:
    print(f"  (removed {_intra_dupes} intra-run duplicate product record(s) from current scrape)")

# ── Merge logic ───────────────────────────────────────────────────────────
if df_archive.empty:
    # First ever run — entire current scrape becomes the archive
    df_archive_updated = df_current.copy()
    print(f"  (First run — {len(df_archive_updated):,} products archived)")
else:
    # 1. Mark all existing records as inactive; re-activate returning ones below
    df_archive = df_archive.copy()
    df_archive["is_active"] = False

    # 2. Split new-scrape rows into truly-new vs. returning
    existing_idx  = df_current.set_index(_MATCH_KEY).index
    archive_idx   = df_archive.set_index(_MATCH_KEY).index
    # FIX: new_mask is a numpy array; removed .values
    new_mask      = ~existing_idx.isin(archive_idx)

    df_new        = df_current[new_mask].copy()        # brand-new products
    df_returning  = df_current[~new_mask].copy()       # returning products

    # 3. Update last_seen_at + is_active for returning products
    df_archive = df_archive.set_index(_MATCH_KEY)
    for _, row in df_returning.iterrows():
        key = (row["brand"], row["product_name"])
        if key in df_archive.index:
            df_archive.at[key, "last_seen_at"] = row["last_seen_at"]
            df_archive.at[key, "is_active"]    = True
    df_archive = df_archive.reset_index()

    # 4. Append brand-new products
    df_archive_updated = pd.concat([df_archive, df_new], ignore_index=True)

# ── Final guard deduplication ─────────────────────────────────────────────
_before_final = len(df_archive_updated)
df_archive_updated = df_archive_updated.drop_duplicates(
    subset=_MATCH_KEY, keep="last"
).reset_index(drop=True)
_final_dupes = _before_final - len(df_archive_updated)
if _final_dupes:
    print(f"  (final guard removed {_final_dupes} duplicate record(s))")

n_active       = df_archive_updated["is_active"].astype(str).str.lower().eq("true").sum()
n_discontinued = len(df_archive_updated) - n_active

print(f"✓ Archive updated:")
print(f"     Total        : {len(df_archive_updated):>5,} product records  (unique by brand + product_name)")
print(f"     Active       : {n_active:>5,}  (seen in this scrape run)")
print(f"     Discontinued : {n_discontinued:>4,}  (absent from this run — possibly discontinued)")

# ── Enrich BRAND_FIBER_LOOKUP with archive-only brands ───────────────────
# BRAND_FIBER_LOOKUP was built from df_catalog (current scrape only) in 1-C-3.
# Brands seen in previous runs but absent from this scrape are missing from it.
# We supplement here so that brand_fiber_lookup.json, written by Save Outputs
# and read by the recommendation model, covers the full historical brand set.
_current_brand_keys = {b.lower().strip() for b in BRAND_FIBER_LOOKUP}
_archive_only_rows  = df_archive_updated[
    ~df_archive_updated["brand"].str.lower().str.strip().isin(_current_brand_keys)
    & df_archive_updated["fiber_json"].notna()
]
_enriched = 0
for _brand, _grp in _archive_only_rows.groupby("brand"):
    _agg: dict = {}
    for _fj in _grp["fiber_json"]:
        try:
            for _k, _v in json.loads(_fj).items():
                _agg.setdefault(_k, []).append(_v)
        except Exception:
            continue
    _brand_fiber = {
        _k: round(float(np.median(_v)), 1)
        for _k, _v in _agg.items()
        if np.median(_v) > 2
    }
    _total = sum(_brand_fiber.values())
    if _total > 0 and _brand_fiber:
        _brand_fiber = {_k: round(_v / _total * 100, 1) for _k, _v in _brand_fiber.items()}
        _key = _brand.lower().strip()
        BRAND_FIBER_LOOKUP[_key] = {
            "brand":       _key,
            "fibers":      _brand_fiber,
            "bio_share":   bio_share(_brand_fiber),
            "biodeg_tier": biodeg_tier(bio_share(_brand_fiber)),
            "item_count":  len(_grp),
            "source":      _grp["source"].iloc[0] if "source" in _grp.columns else "archive",
        }
        _enriched += 1

if _enriched:
    print(f"\n  BRAND_FIBER_LOOKUP enriched with +{_enriched} archive-only brand(s)")
print(f"  BRAND_FIBER_LOOKUP total: {len(BRAND_FIBER_LOOKUP)} brands  "
      f"(current scrape + historical archive)")
print(f"  → Save Outputs will write this enriched lookup to brand_fiber_lookup.json")


✓ No existing archive — initialising fresh archive.
  (First run — 1,381 products archived)
✓ Archive updated:
     Total        : 1,381 product records  (unique by brand + product_name)
     Active       : 1,381  (seen in this scrape run)
     Discontinued :    0  (absent from this run — possibly discontinued)
  BRAND_FIBER_LOOKUP total: 31 brands  (current scrape + historical archive)
  → Save Outputs will write this enriched lookup to brand_fiber_lookup.json


In [15]:
# ── 1-D  Save updated archive (timestamped + stable alias) ──────────────────
# _ts is defined here so this cell can run independently before Save Outputs.
_ts             = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
archive_ts_path = WEB_DIR / f"{_ts}-webscraped_catalog_archive.csv"

df_archive_updated.to_csv(ARCHIVE_PATH,    index=False, encoding="utf-8")
df_archive_updated.to_csv(archive_ts_path, index=False, encoding="utf-8")

print(f"✓ webscraped_catalog_archive.csv  (stable alias)  →  {ARCHIVE_PATH}")
print(f"✓ {archive_ts_path.name}          →  {archive_ts_path}")
print(f"  {len(df_archive_updated):,} total records across all scrape runs")
print(f"  Columns: {list(df_archive_updated.columns)}")


✓ webscraped_catalog_archive.csv  (stable alias)  →  D:\School\ISPROJ2\weaveforward-ml\data\webscraped_data\webscraped_catalog_archive.csv
✓ 20260424-032653-webscraped_catalog_archive.csv          →  D:\School\ISPROJ2\weaveforward-ml\data\webscraped_data\20260424-032653-webscraped_catalog_archive.csv
  1,381 total records across all scrape runs
  Columns: ['brand', 'product_name', 'clothing_type', 'fabric_composition', 'fiber_json', 'most_dominant_fiber', 'fs_bio_share', 'fs_biodeg_tier', 'source', 'country_of_brand', 'scraped_url', 'first_scraped_at', 'last_seen_at', 'origin', 'is_active']


---
## 1-E · Fiber Approximation — Discontinued / Out-of-Catalogue Items

When a donor submits a garment whose `(brand, clothing_type)` combination is no
longer present in the live product catalogue (i.e. the item has been discontinued
or the brand has not been scraped), `fiber_approximation()` infers the fiber
composition from the **historical archive** using a **greedy priority-ordered
nearest-neighbour lookup**.

> **Why `fiber_approximation` and not `fiber_match`?**  
> `fiber_match` / `is_match` is the binary classification performed by CatBoost
> in the recommendation model (Section 5–8 of the main notebook). This function
> only *approximates* a plausible fiber profile for a garment that is absent from
> the live catalogue — it does not route or score donations.

| Tier | Condition | Strategy |
|---|---|---|
| **1** | Exact brand + clothing_type found in archive | Most recently seen product record |
| **2** | Brand found in archive, any clothing type | Most recently seen product from that brand |
| **3** | Clothing type found in archive, any brand | Median fiber profile across all archived items of that category |
| **4** | Neither brand nor type in archive | Global median fiber profile from the entire archive |

**Why greedy?** The function unconditionally accepts the **first tier** that
returns a match — no cost comparison across tiers, no backtracking. Resolution
is deterministic and auditable: every result carries `approx_tier` (1–4) and
`approx_reason` for traceability in downstream feature engineering.

The returned dict (`fiber_json`, `fiber_dict`, `most_dominant_fiber`,
`fs_bio_share`, `fs_biodeg_tier`) has the **same schema** as `brand_fiber_lookup.json`,
so it slots directly into Section 5-B-1 of the main ML notebook without any
changes to the downstream pipeline.


In [16]:
# ── 1-E  Helper utilities for fiber_approximation() ──────────────────────

def _parse_fiber_json_safe(val) -> dict:
    """Parse a fiber_json string to a dict; return {} on any error."""
    try:
        return json.loads(val) if isinstance(val, str) else {}
    except (json.JSONDecodeError, TypeError):
        return {}


def _compute_median_fiber_profile(subset: pd.DataFrame) -> dict:
    """
    Compute the median per-fiber percentage across a subset of archive rows.
    Fibers with median share ≤ 2 % are dropped; the remainder is
    re-normalised to 100 %.
    """
    agg: dict = {}
    for _, row in subset.iterrows():
        for fib, pct in _parse_fiber_json_safe(row["fiber_json"]).items():
            agg.setdefault(fib, []).append(pct)
    profile = {
        k: round(float(np.median(v)), 1)
        for k, v in agg.items()
        if np.median(v) > 2
    }
    total = sum(profile.values())
    if total > 0:
        profile = {k: round(v / total * 100, 1) for k, v in profile.items()}
    return profile


# ── 1-E  Fiber approximation for discontinued/out-of-catalogue items ──────
# NOTE: This function infers a plausible fiber profile from the historical
# archive.  It is intentionally named fiber_approximation (not fiber_match)
# to distinguish it from the is_match binary classification in the
# recommendation model (catboost_fiber_match.cbm, Sections 5-8).

def fiber_approximation(
    brand: str,
    clothing_type: str,
    archive_df: pd.DataFrame,
    prefer_active: bool = True,
) -> dict:
    """
    Approximate fiber composition for a (brand, clothing_type) pair that is
    absent from the current live catalogue.

    Uses a greedy priority-ordered nearest-neighbour lookup against the
    historical archive (webscraped_catalog_archive.csv):

        Tier 1 — exact brand + clothing_type match  (most recently seen row)
        Tier 2 — brand match, any clothing type      (most recently seen row)
        Tier 3 — clothing_type match, any brand      (median fiber profile)
        Tier 4 — no brand or type match              (global median profile)

    Parameters
    ----------
    brand         : Brand name string (case-insensitive).
    clothing_type : Clothing category string (case-insensitive).
    archive_df    : Historical archive DataFrame (webscraped_catalog_archive.csv).
    prefer_active : If True, prefer currently active items within each tier
                    before falling back to discontinued archive records.

    Returns
    -------
    dict with keys:
        fiber_json             — JSON string of approximated fiber composition
        fiber_dict             — Python dict of { fiber: pct }
        most_dominant_fiber    — fiber with highest share
        fs_bio_share           — bio-fiber share percentage
        fs_biodeg_tier         — 'high' / 'medium' / 'low'
        approx_tier            — int 1–4 (which tier resolved the query)
        approx_reason          — human-readable description of the resolution
        matched_brand          — actual brand used in the approximation
        matched_clothing_type  — actual clothing_type used in the approximation
    """

    def _best_row(subset: pd.DataFrame) -> pd.Series:
        """Return the most recently seen row; prefer active items if requested."""
        if prefer_active:
            active = subset[subset["is_active"].astype(str).str.lower() == "true"]
            if not active.empty:
                return active.sort_values("last_seen_at", ascending=False).iloc[0]
        return subset.sort_values("last_seen_at", ascending=False).iloc[0]

    def _result_from_row(row: pd.Series, tier: int, reason: str) -> dict:
        fibers = _parse_fiber_json_safe(row["fiber_json"])
        bs     = bio_share(fibers)
        return {
            "fiber_json":             row["fiber_json"],
            "fiber_dict":             fibers,
            "most_dominant_fiber":    max(fibers, key=fibers.get) if fibers else "unknown",
            "fs_bio_share":           bs,
            "fs_biodeg_tier":         biodeg_tier(bs),
            "approx_tier":            tier,
            "approx_reason":          reason,
            "matched_brand":          row["brand"],
            "matched_clothing_type":  row["clothing_type"],
        }

    def _result_from_profile(
        profile: dict, ref_df: pd.DataFrame, tier: int, reason: str
    ) -> dict:
        bs        = bio_share(profile)
        fj        = json.dumps(profile)
        dom_fiber = max(profile, key=profile.get) if profile else "unknown"
        return {
            "fiber_json":             fj,
            "fiber_dict":             profile,
            "most_dominant_fiber":    dom_fiber,
            "fs_bio_share":           bs,
            "fs_biodeg_tier":         biodeg_tier(bs),
            "approx_tier":            tier,
            "approx_reason":          reason,
            "matched_brand":          ref_df["brand"].iloc[0] if tier == 3 else "global_median",
            "matched_clothing_type":  ref_df["clothing_type"].iloc[0] if tier == 3 else "all",
        }

    # ── Normalise inputs ──────────────────────────────────────────────────
    b_norm  = brand.lower().strip()
    ct_norm = clothing_type.lower().strip()

    df = archive_df.copy()
    df["_b"]  = df["brand"].str.lower().str.strip()
    df["_ct"] = df["clothing_type"].str.lower().str.strip()

    # ── Tier 1: exact brand + clothing_type ───────────────────────────────
    t1 = df[(df["_b"] == b_norm) & (df["_ct"] == ct_norm)]
    if not t1.empty:
        return _result_from_row(
            _best_row(t1), 1,
            f"exact match — brand '{brand}' + clothing_type '{clothing_type}' found in archive",
        )

    # ── Tier 2: same brand, any clothing type ─────────────────────────────
    t2 = df[df["_b"] == b_norm]
    if not t2.empty:
        return _result_from_row(
            _best_row(t2), 2,
            f"brand match — '{brand}' found in archive (clothing_type '{clothing_type}' not catalogued)",
        )

    # ── Tier 3: any brand, same clothing type — median profile ────────────
    t3 = df[df["_ct"] == ct_norm]
    if not t3.empty:
        profile = _compute_median_fiber_profile(t3)
        return _result_from_profile(
            profile, t3, 3,
            f"clothing_type median — '{clothing_type}' found across "
            f"{t3['brand'].nunique()} brand(s) in archive (brand '{brand}' not catalogued)",
        )

    # ── Tier 4: global median profile ─────────────────────────────────────
    profile = _compute_median_fiber_profile(df)
    return _result_from_profile(
        profile, df, 4,
        f"global median — brand '{brand}' and clothing_type '{clothing_type}' "
        f"both absent from archive ({len(df):,} records used)",
    )


print("✓ fiber_approximation() ready")
print("  Tier resolution order:")
print("    1 → exact brand + clothing_type (archive)")
print("    2 → brand match, any clothing type")
print("    3 → clothing_type median across all brands")
print("    4 → global median fiber profile")
print()
print("  Return dict keys: fiber_json | fiber_dict | most_dominant_fiber |")
print("                    fs_bio_share | fs_biodeg_tier | approx_tier |")
print("                    approx_reason | matched_brand | matched_clothing_type")


✓ fiber_approximation() ready
  Tier resolution order:
    1 → exact brand + clothing_type (archive)
    2 → brand match, any clothing type
    3 → clothing_type median across all brands
    4 → global median fiber profile

  Return dict keys: fiber_json | fiber_dict | most_dominant_fiber |
                    fs_bio_share | fs_biodeg_tier | approx_tier |
                    approx_reason | matched_brand | matched_clothing_type


---
## Save Outputs

In [17]:
# ── Save webscraped catalog CSV — timestamped ─────────────────────────────
_ts              = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
catalog_csv      = WEB_DIR / f"{_ts}-webscraped_catalog.csv"
catalog_csv_stable = WEB_DIR / "webscraped_catalog.csv"   # stable alias

_save_cols = [
    "brand", "product_name", "clothing_type", "fabric_composition",
    "fiber_json", "most_dominant_fiber", "fs_bio_share", "fs_biodeg_tier",
    "source", "country_of_brand", "scraped_url", "scraped_at", "origin",
]
df_catalog[_save_cols].to_csv(catalog_csv,        index=False, encoding="utf-8")
df_catalog[_save_cols].to_csv(catalog_csv_stable, index=False, encoding="utf-8")

print(f"✓ {catalog_csv.name}  →  {catalog_csv}")
print(f"✓ webscraped_catalog.csv (stable alias)  →  {catalog_csv_stable}")
print(f"  {len(df_catalog):,} rows | {df_catalog['brand'].nunique()} brands | "
      f"{len(_save_cols)} columns")


✓ 20260424-032653-webscraped_catalog.csv  →  D:\School\ISPROJ2\weaveforward-ml\data\webscraped_data\20260424-032653-webscraped_catalog.csv
✓ webscraped_catalog.csv (stable alias)  →  D:\School\ISPROJ2\weaveforward-ml\data\webscraped_data\webscraped_catalog.csv
  1,381 rows | 31 brands | 13 columns


### Save BRAND_FIBER_LOOKUP

Writes `brand_fiber_lookup.json` to `data/processed/` for backward compatibility
with the main classification notebook.

In [18]:
# ── Save BRAND_FIBER_LOOKUP JSON ─────────────
lookup_path        = PROC_DIR / f"{_ts}-brand_fiber_lookup.json"
lookup_path_stable = PROC_DIR / "brand_fiber_lookup.json"

for path in [lookup_path, lookup_path_stable]:
    with open(path, "w") as f:
        json.dump(BRAND_FIBER_LOOKUP, f, indent=2)

print(f"✓ {lookup_path.name}  →  {lookup_path}")
print(f"✓ brand_fiber_lookup.json (stable alias)  →  {lookup_path_stable}")
print(f"  {len(BRAND_FIBER_LOOKUP):,} brand entries")

print(f"\n── Webscraper extraction complete — run ID: {_ts} ─────────────────")
print(f"   Next: load  data/webscraped_data/{catalog_csv.name}")
print(f"         into  weaveforward_fiber_recommendation.ipynb")


✓ 20260424-032653-brand_fiber_lookup.json  →  D:\School\ISPROJ2\weaveforward-ml\data\processed\20260424-032653-brand_fiber_lookup.json
✓ brand_fiber_lookup.json (stable alias)  →  D:\School\ISPROJ2\weaveforward-ml\data\processed\brand_fiber_lookup.json
  31 brand entries

── Webscraper extraction complete — run ID: 20260424-032653 ─────────────────
   Next: load  data/webscraped_data/20260424-032653-webscraped_catalog.csv
         into  weaveforward_fiber_recommendation.ipynb
